In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/devlucascfarias/logos-3.git"
REPO_BRANCH = "main"
WORKDIR = "/content/logos-3"
repo = pathlib.Path(WORKDIR)

token = userdata.get("GH_TOKEN")
git = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

if (repo / ".git").exists():
    subprocess.run(git + ["-C", WORKDIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
elif repo.exists():
    raise RuntimeError(f"{WORKDIR} existe, mas não é um repositório Git")
else:
    subprocess.run(git + ["clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, WORKDIR], check=True)

token = auth = None
os.chdir(WORKDIR)
print(f"Repositório sincronizado em {os.getcwd()}")

# Qwen3-8B QLoRA em NVIDIA L4

Continuação isolada do piloto de 500k. Reutiliza exatamente os dados aprovados e parte do adapter preservado no Drive, sem sobrescrever o experimento original.

In [ ]:
STAGE = "pilot_continuation"
DATA_STAGE = "pilot"  # reutiliza data/processed/pilot
RUN_DATA_PREPARATION = False
RUN_TRAINING = True
FRESH_RUN = True  # não retoma uma continuação anterior

PRESERVED_RUN_PATH = "/content/drive/MyDrive/logos-3/runs/pilot_500k_step7"
SOURCE_ADAPTER_PATH = f"{PRESERVED_RUN_PATH}/adapter"
DATA_BACKUP_PATH = f"{PRESERVED_RUN_PATH}/data"
REFERENCE_ADAPTER_PATH = SOURCE_ADAPTER_PATH

# Usados apenas se RUN_DATA_PREPARATION=True.
TOKEN_BUDGET = None
MAX_SOURCE_ROWS = None
MAX_STEPS = None
MAX_TRAIN_SAMPLES = None
EVAL_SEED = 20260722
print({"stage": STAGE, "data_stage": DATA_STAGE, "fresh": FRESH_RUN, "source_adapter": SOURCE_ADAPTER_PATH})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN não definido; apenas fontes públicas sem aceite funcionarão.")
hf_token = None

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/environment_check.py"], check=True)

In [ ]:
import hashlib, json, pathlib, shutil, subprocess, sys
from google.colab import drive

if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

if RUN_DATA_PREPARATION:
    command = [sys.executable, "scripts/prepare_data.py", "--stage", DATA_STAGE]
    if TOKEN_BUDGET is not None:
        command += ["--token-budget", str(TOKEN_BUDGET)]
    if MAX_SOURCE_ROWS is not None:
        command += ["--max-source-rows", str(MAX_SOURCE_ROWS)]
    subprocess.run(command, check=True)

local_data = pathlib.Path(WORKDIR) / "data" / "processed" / DATA_STAGE
backup_data = pathlib.Path(DATA_BACKUP_PATH)
source_manifest_path = pathlib.Path(SOURCE_ADAPTER_PATH) / "run_manifest.json"
if not source_manifest_path.exists():
    raise FileNotFoundError(f"Manifesto do piloto ausente: {source_manifest_path}")
source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
expected_hashes = {
    "train.jsonl": source_manifest["data"]["train_sha256"],
    "validation.jsonl": source_manifest["data"]["validation_sha256"],
    "dataset_report.json": source_manifest["data"]["dataset_report_sha256"],
}
for name, expected_hash in expected_hashes.items():
    local_path = local_data / name
    backup_path = backup_data / name
    backup_valid = (
        backup_path.exists()
        and backup_path.stat().st_size > 0
        and hashlib.sha256(backup_path.read_bytes()).hexdigest() == expected_hash
    )
    if (not local_path.exists() or local_path.stat().st_size == 0) and backup_valid:
        local_data.mkdir(parents=True, exist_ok=True)
        shutil.copy2(backup_path, local_path)
        print(f"Restaurado do Drive: {local_path}")
    if not local_path.exists() or local_path.stat().st_size == 0:
        raise FileNotFoundError(f"Dado aprovado ausente: {local_path}")
    actual_hash = hashlib.sha256(local_path.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"Hash divergente para {local_path}")
    if not backup_valid:
        backup_data.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, backup_path)
        print(f"Backup criado no Drive: {backup_path}")
print("Dados pilot verificados por hash; preprocessing não será repetido.")

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
import codecs, datetime, os, pathlib, shutil, subprocess, sys
from google.colab import drive

if SOURCE_ADAPTER_PATH.startswith("/content/drive/") and not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
source_adapter = pathlib.Path(SOURCE_ADAPTER_PATH)
for required_name in ("adapter_config.json", "adapter_model.safetensors"):
    required_path = source_adapter / required_name
    if not required_path.exists() or required_path.stat().st_size == 0:
        raise FileNotFoundError(f"Adapter inicial incompleto: {required_path}")

if RUN_TRAINING:
    fresh_run = FRESH_RUN
    if fresh_run:
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_root = pathlib.Path(WORKDIR) / "outputs" / "archive" / f"{STAGE}-{timestamp}"
        old_outputs = {
            "checkpoints": pathlib.Path(WORKDIR) / "outputs" / "checkpoints" / STAGE,
            "adapter": pathlib.Path(WORKDIR) / "outputs" / "adapters" / STAGE,
        }
        for label, source in old_outputs.items():
            if source.exists():
                archive_root.mkdir(parents=True, exist_ok=True)
                shutil.move(str(source), str(archive_root / label))
                print(f"Arquivado: {source} -> {archive_root / label}")
        FRESH_RUN = False
    command = [
        sys.executable, "-u", "scripts/train_sft.py",
        "--stage", STAGE,
        "--data-stage", DATA_STAGE,
        "--adapter-path", SOURCE_ADAPTER_PATH,
    ]
    if not fresh_run:
        command += ["--resume-from-checkpoint", "auto"]
    if MAX_STEPS is not None:
        command += ["--max-steps", str(MAX_STEPS)]
    if MAX_TRAIN_SAMPLES is not None:
        command += ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
    print("Iniciando treino com barra de progresso e ETA...", flush=True)
    log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_train.log")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    process = subprocess.Popen(
        command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    assert process.stdout is not None
    with open(log_path, "wb") as log_file:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            log_file.write(chunk)
            log_file.flush()
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b"", final=True))
        sys.stdout.flush()
    return_code = process.wait()
    print(f"\nLog salvo em: {log_path}", flush=True)
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("RUN_TRAINING=False: dados e testes prontos; treino não iniciado.")

## Comparação cega: base vs. piloto 500k vs. continuação

Esta etapa compara a continuação com o adapter original preservado no Drive e com o modelo-base. As identidades são embaralhadas como A/B/C; avalie `comparison.md` antes de abrir `mapping.json`.

In [ ]:
import codecs, os, pathlib, subprocess, sys

if REFERENCE_ADAPTER_PATH.startswith("/content/drive/") and not pathlib.Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

EVAL_OUTPUT_DIR = "outputs/evaluations/pilot_continuation"
command = [
    sys.executable, "-u", "scripts/compare_adapter.py",
    "--stage", STAGE,
    "--output-dir", EVAL_OUTPUT_DIR,
    "--seed", str(EVAL_SEED),
]
if REFERENCE_ADAPTER_PATH and pathlib.Path(REFERENCE_ADAPTER_PATH).exists():
    command += ["--reference-adapter-path", REFERENCE_ADAPTER_PATH]
else:
    raise FileNotFoundError(f"Adapter de referência não encontrado: {REFERENCE_ADAPTER_PATH}")
print("Iniciando comparação cega com progresso...", flush=True)
log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_comparison.log")
os.makedirs(os.path.dirname(log_path), exist_ok=True)
process = subprocess.Popen(
    command,
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
assert process.stdout is not None
with open(log_path, "wb") as log_file:
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        log_file.write(chunk)
        log_file.flush()
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b"", final=True))
    sys.stdout.flush()
return_code = process.wait()
print(f"\nLog salvo em: {log_path}", flush=True)
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

comparison_path = Path(WORKDIR) / EVAL_OUTPUT_DIR / "comparison.md"
display(Markdown(comparison_path.read_text(encoding="utf-8")))
print("Avalie A/B/C acima antes de abrir mapping.json.")
print("Planilha de notas:", Path(WORKDIR) / EVAL_OUTPUT_DIR / "ratings.json")

## Depois da avaliação

Somente após atribuir as notas, abra `outputs/evaluations/pilot_continuation/mapping.json`. A continuação só substitui o piloto preservado se vencer comportamentalmente, não apenas pelo `eval_loss`.